# Phase 2 — Chuyển mask → COCO

Notebook này **mỏng**: nó chỉ gọi `src/floodcount/data/mask_to_coco.py` trong
repo, không chứa logic. Sửa logic thì sửa file `.py` rồi `git pull`, không sửa
trong notebook — để code còn tái lập được và không bị lệch giữa các phiên.

**Không cần GPU cho Phase 2.** Bước này chỉ đọc ảnh, resize, tách component và
ghi file bằng CPU. Chạy ở runtime CPU cũng được, khỏi tốn suất T4.

**Không cần cài MMDetection.** Chỉ dùng `cv2`, `numpy`, `yaml`.

Kết quả của bước này là `floodnet_coco.zip` (~1–2 GB thay vì 13 GB), để Phase 3
copy sang `/content` nhanh.

## ⚠️ Chạy lần lượt từ trên xuống, bắt đầu từ ô `[2.1] CẤU HÌNH`

Colab chạy **từng ô độc lập**, không tự chạy ô phía trên. Nhảy thẳng xuống ô
`[2.6]` sẽ lỗi `NameError: name 'PROCESSED_DIR' is not defined` — nghĩa là ô
`[2.1]` chưa chạy, không phải lỗi code.

In [ ]:
# [2.1] CẤU HÌNH — sửa ở đây, không sửa trong code
REPO_URL      = "https://github.com/21xDHoang/flood-house-counting.git"
REPO_DIR      = "/content/flood-house-counting"

DRIVE_DIR     = "/content/drive/MyDrive/Flood_House_AI"
ZIP_PATH      = f"{DRIVE_DIR}/floodnet_raw.zip"      # 13 GB, đã có sẵn trên Drive
BUILD_OUT     = f"{DRIVE_DIR}/runs/build"            # báo cáo + ảnh overlay GT

# Dataset dựng trên /content (đĩa cục bộ) rồi mới nén đẩy lên Drive.
# Ghi thẳng 4686 file lên Drive qua FUSE sẽ chậm hơn nhiều lần.
PROCESSED_DIR = "/content/floodnet_coco"
PROCESSED_ZIP = f"{DRIVE_DIR}/processed/floodnet_coco.zip"

# Thư mục trung gian cho lần CHẠY THỬ, tách riêng để không lẫn vào kết quả thật
WORK_DIR_THU  = "/content/floodnet_work_thu"

print("Đã nạp cấu hình Phase 2.")
print(f"  repo     : {REPO_URL}")
print(f"  zip gốc  : {ZIP_PATH}")
print(f"  dataset  : {PROCESSED_DIR}")
print(f"  báo cáo  : {BUILD_OUT}")

In [ ]:
# [2.2] GẮN GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# [2.3] LẤY CODE TỪ GITHUB
#
# /content mất sạch mỗi lần runtime reset (xem docs/NOTES.md muc 1.6), nên phiên
# nào cũng phải làm bước này. Có sẵn repo rồi thì `git pull` cho nhanh.
import os
import subprocess


def chay(lenh, **them):
    kq = subprocess.run(lenh, capture_output=True, text=True, **them)
    if kq.stdout.strip():
        print(kq.stdout.strip())
    if kq.returncode != 0:
        print("LỖI:", kq.stderr.strip())
    return kq.returncode


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Đã có repo, cập nhật code mới nhất:")
    chay(["git", "-C", REPO_DIR, "pull"])
else:
    print("Chưa có repo, clone về:")
    chay(["git", "clone", REPO_URL, REPO_DIR])

print()
print("Commit đang dùng:")
chay(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])

# Kiểm repo đã có đủ code Phase 2 chưa. Clone từ lúc chưa push sẽ thiếu configs/,
# và script khi đó thoát ngay mà không in gì — rất khó đoán ra.
print()
print("Kiểm tra code Phase 2:")
for duong_dan_can in ("configs/data.yaml", "scripts/build_coco.py",
                      "src/floodcount/data/mask_to_coco.py",
                      "src/floodcount/data/resize.py"):
    co = os.path.exists(os.path.join(REPO_DIR, duong_dan_can))
    print(f"  [{'OK ' if co else 'THIẾU'}] {duong_dan_can}")
    if not co:
        print("  -> Thiếu code. Xoá thư mục repo rồi chạy lại ô này để clone lại.")
        break

In [ ]:
# [2.4] CHUẨN BỊ: kiểm thư viện + hàm chạy lệnh
import importlib
import subprocess
import sys

for ten_module, ten_goi_pip in [("cv2", "opencv-python"),
                                ("yaml", "pyyaml"),
                                ("numpy", "numpy")]:
    try:
        m = importlib.import_module(ten_module)
        print(f"{ten_module:<6}: {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{ten_module:<6}: THIẾU -> đang cài {ten_goi_pip} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", ten_goi_pip])


def chay_hien_dan(lenh, cwd):
    """Chạy lệnh con và in từng dòng NGAY khi nó nhả ra, trả về mã thoát.

    KHÔNG dùng thẳng `subprocess.run(lenh, cwd=...)` cho các bước chạy dài.
    Trên Colab, fd 1 của tiến trình con không chảy vào ô output: script chạy
    xong, mã thoát 0, mà ô output trống trơn, không biết nó làm được gì. Đã
    dính ba lần ngày 30/09/2026 mới nhận ra quy luật.

    Phải tự đọc ống dẫn rồi in qua stdout của kernel thì mới thấy. Cách này
    vẫn giữ được log hiện dần chứ không dồn tới lúc chạy xong.

    Đối số `-u` truyền cho python của tiến trình con là bắt buộc: không có nó
    thì chính tiến trình con lại đệm output theo khối, và ta ngồi nhìn màn
    hình im lặng suốt vài phút rồi output mới ập ra một lúc.
    """
    kq = subprocess.Popen(lenh, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for dong in kq.stdout:
        print(dong, end="")
    kq.wait()
    return kq.returncode


print("\nSẵn sàng.")

---
## Bước 1 — Dựng thử 3 ảnh mỗi split

**Chạy ô này trước, đừng chạy thẳng ô đầy đủ.**

Chạy thử mất ~30 giây; nếu có gì sai (cấu trúc zip khác, config thiếu khoá,
quyền ghi Drive) thì biết ngay, thay vì mất ~20 phút mới phát hiện.

Ô này ghi vào thư mục **riêng** (`PROCESSED_DIR + "_thu"`), không đụng vào kết
quả thật.

In [ ]:
# [2.5] DỰNG THỬ — 3 ảnh mỗi split
#
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [2.1]-[2.4]. "
                     "Hãy chạy lần lượt từ ô [2.1] trở xuống rồi thử lại.")

import sys

# `-u` để tiến trình con không đệm output — xem giải thích ở ô [2.4].
lenh_thu = [sys.executable, "-u", "scripts/build_coco.py",
            "--config", "configs/data.yaml",
            "--max-images", "3",
            "--work-dir", WORK_DIR_THU,
            "--processed-dir", PROCESSED_DIR + "_thu",
            "--report-dir", BUILD_OUT + "_thu",
            "--fresh"]
print("Chạy: " + " ".join(lenh_thu))
print("=" * 72)
ma_thu = chay_hien_dan(lenh_thu, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_thu, "(0 là thành công)")

**Đọc kết quả chạy thử:**

- Dòng `Đối chiếu Phase 1` sẽ ghi **BỎ QUA** — đúng như dự kiến. Phép đối chiếu
  chỉ chạy khi có `audit.jsonl` của Phase 1 **trong cùng phiên**, mà ô [2.5] lại
  dùng `WORK_DIR_THU` riêng. Muốn thấy phép kiểm này ngay thì chạy ô [2.6]
  (ô đó dùng chung `work_dir` với Phase 1).
- Số ảnh mỗi split phải là **3** (hoặc ít hơn nếu split đó có ít ảnh hơn).
- Dòng `Kiểm cấu trúc COCO` phải ghi **ĐẠT**.
- Nếu có dòng `⚠️ Có cặp ảnh/mask bị bỏ` thì **dừng lại**, gửi nguyên phần đó
  cho Claude. Ảnh và mask lệch kích thước là chuyện Phase 1 để ngỏ, phải xem
  mới biết xử lý thế nào.

Chạy thử xong mà ổn thì mới chạy ô tiếp theo.

---
## Bước 2 — Chạy đầy đủ

Toàn bộ 2343 ảnh: resize, tách component, ghi JPEG + PNG mask, ráp COCO JSON.
Mất khoảng **15–30 phút**.

Ô này **tự chạy tiếp nếu bị ngắt**: kết quả từng ảnh được ghi ngay xuống đĩa,
chạy lại sẽ bỏ qua ảnh đã xử lý. Colab ngắt giữa chừng thì cứ chạy lại ô này.

**Chạy ô [1.6] của notebook 01 trong cùng phiên trước ô này** nếu muốn thấy phép
đối chiếu Phase 1 — nó cần `audit.jsonl` còn nằm trong `/content`.

In [ ]:
# [2.6] CHẠY ĐẦY ĐỦ
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [2.1]-[2.4]. "
                     "Hãy chạy lần lượt từ ô [2.1] trở xuống rồi thử lại.")

import sys

lenh_day_du = [sys.executable, "-u", "scripts/build_coco.py",
               "--config", "configs/data.yaml",
               "--processed-dir", PROCESSED_DIR,
               "--report-dir", BUILD_OUT]
print("Chạy: " + " ".join(lenh_day_du))
print("=" * 72)
ma_day_du = chay_hien_dan(lenh_day_du, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_day_du, "(0 là thành công)")

In [ ]:
# [2.7] XEM BÁO CÁO
if "BUILD_OUT" not in globals():
    raise SystemExit("Chưa chạy ô [2.1] CẤU HÌNH. "
                     "Hãy chạy lần lượt từ ô [2.1] trở xuống rồi thử lại.")

import os

# Ưu tiên báo cáo chạy đầy đủ. Chưa có thì lấy tạm báo cáo chạy thử, nhưng phải
# NÓI RÕ là số liệu chưa đầy đủ — nếu không, rất dễ đọc nhầm số của 3 ảnh/split.
duong_day_du = f"{BUILD_OUT}/BUILD_REPORT.md"
duong_chay_thu = f"{BUILD_OUT}_thu/BUILD_REPORT.md"

if os.path.exists(duong_day_du):
    thu_muc_bao_cao, nhan = BUILD_OUT, "CHẠY ĐẦY ĐỦ"
elif os.path.exists(duong_chay_thu):
    thu_muc_bao_cao, nhan = BUILD_OUT + "_thu", "CHẠY THỬ — số liệu CHƯA đầy đủ"
else:
    raise SystemExit("Chưa có báo cáo nào. Đã tìm:\n"
                     f"  {duong_day_du}\n  {duong_chay_thu}\n"
                     "Hãy chạy ô [2.5] hoặc [2.6] trước.")

print(f"===== Báo cáo của lần {nhan} =====")
print(open(f"{thu_muc_bao_cao}/BUILD_REPORT.md", encoding="utf-8").read())

thu_muc_overlay = f"{thu_muc_bao_cao}/overlay_gt"
ds_anh = sorted(os.listdir(thu_muc_overlay)) if os.path.isdir(thu_muc_overlay) else []
print(f"\nẢnh overlay GT ({len(ds_anh)} tệp) trong {thu_muc_overlay}:")
for t in ds_anh:
    print("  " + t)

---
## Bước 3 — Nén và lưu lên Drive

Chỉ chạy sau khi ô [2.7] đã **sạch**: không có dòng ⚠️ bỏ cặp nào, `Kiểm cấu trúc
COCO` ĐẠT, và `Đối chiếu Phase 1` ghi **KHỚP HOÀN TOÀN**.

Nén bằng chế độ **lưu thẳng, không nén lại** (`ZIP_STORED`): ảnh JPEG và mask PNG
vốn đã được nén sẵn, thử nén thêm lần nữa chỉ thu được ~0% mà tốn vài phút CPU.

Bước này cần **~1–2 GB trống trên Drive**. Nếu Drive sắp đầy thì đặt
`build_coco.luu_mask: false` trong `configs/data.yaml` rồi chạy lại từ ô [2.6].

In [ ]:
# [2.8] NÉN VÀ LƯU LÊN DRIVE
if "PROCESSED_DIR" not in globals():
    raise SystemExit("Chưa chạy ô [2.1] CẤU HÌNH. "
                     "Hãy chạy lần lượt từ ô [2.1] trở xuống rồi thử lại.")

import os
import zipfile

if not os.path.isdir(PROCESSED_DIR):
    raise SystemExit(f"Chưa có dataset: {PROCESSED_DIR}\n"
                     "Hãy chạy ô [2.6] trước.")

os.makedirs(os.path.dirname(PROCESSED_ZIP), exist_ok=True)

so_file = 0
with zipfile.ZipFile(PROCESSED_ZIP, "w", zipfile.ZIP_STORED) as z:
    for thu_muc, _, ds_ten in os.walk(PROCESSED_DIR):
        for ten in ds_ten:
            duong_dan = os.path.join(thu_muc, ten)
            z.write(duong_dan, os.path.relpath(duong_dan, PROCESSED_DIR))
            so_file += 1
            if so_file % 500 == 0:
                print(f"  ... đã nén {so_file} tệp")

print(f"\nXong: {so_file} tệp -> {PROCESSED_ZIP}")
print(f"Dung lượng: {os.path.getsize(PROCESSED_ZIP) / 1024**3:.2f} GB")
print("\nTừ giờ Phase 3 chỉ cần giải nén tệp này vào /content, không cần tới")
print("floodnet_raw.zip 13 GB nữa.")

---
## Gửi kết quả cho Claude

Copy 2 thứ này:

1. **Nội dung `BUILD_REPORT.md`** ở ô [2.7].
2. **Nhận xét bằng mắt** sau khi mở thư mục `overlay_gt` trên Drive:
   - Box đỏ (nhà ngập) và xanh (nhà không ngập) có **khớp với nhà** không?
   - Số ghi ở góc ảnh có **đúng bằng số box** nhìn thấy không?
   - Có nhà nào **bị bỏ sót** (thấy nhà mà không có box) không?

Phép kiểm này quan trọng vì nó soi **đúng thứ model sẽ đọc lúc train**: file JSON
và ảnh JPEG đã ghi ra đĩa, chứ không phải mask gốc.

⚠️ **Đừng quên bước nén ở ô [2.8].** Không nén thì lần sau Colab reset là mất
sạch 20 phút vừa chạy.